## Section 1. Gather

In [16]:
import glob
import getpass
import hashlib
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

START_YEAR, END_YEAR = 2010, 2026          
PORTAL_YEARS = range(2021, END_YEAR + 1)   
PAUSE_SECONDS = 1.5                        

BASE_URL = "https://api.collegefootballdata.com"
USER_AGENT = ("INFO4617-class-project/1.0 "
              "(CU Boulder NFL rookie ages - contact: zezw9901@colorado.edu)")

DATA = Path("data")
RAW_DIR = DATA / "raw_cfbd"      
CLEAN_DIR = DATA / "clean"       
SAMPLE_DIR = DATA / "sample"     
for folder in (RAW_DIR, CLEAN_DIR, SAMPLE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

### 1.2 key

In [17]:
API_KEY = os.environ.get("CFBD_API_KEY") or getpass.getpass("Paste your CFBD API key (not saved): ")

Paste your CFBD API key (not saved):  ········


### 1.3 Request

In [31]:
_last_call = 0.0
def polite_get(path, params=None, retries=3):
    """GET one CFBD endpoint, pausing before each request."""
    global _last_call
    for attempt in range(1, retries + 1):
        wait = PAUSE_SECONDS - (time.monotonic() - _last_call)
        if wait > 0:
            time.sleep(wait)
        _last_call = time.monotonic()

        resp = requests.get(
            BASE_URL + path, params=params, timeout=60,
            headers={"Authorization": f"Bearer {API_KEY}",
                     "User-Agent": USER_AGENT,
                     "Accept": "application/json"},
        )
        if resp.status_code == 401:
            raise RuntimeError("HTTP 401: the API key was rejected. Check the key from the CFBD email.")
        if resp.status_code == 429 or resp.status_code >= 500:
            print(f"  HTTP {resp.status_code} on {path} {params}; waiting before retry {attempt}/{retries}")
            time.sleep(PAUSE_SECONDS * 2 ** attempt)
            continue
        resp.raise_for_status()
        return resp
    raise RuntimeError(f"Gave up on {path} {params}. If it was 429, the free monthly call limit may be used up.")

### 1.4 Draft picks
Each response is saved untouched as `data/raw_cfbd/draft_picks_<year>.json`.

In [19]:
for year in range(START_YEAR, END_YEAR + 1):
    out = RAW_DIR / f"draft_picks_{year}.json"
    if out.exists():
        print(f"{year}: already saved, skipping")
        continue
    resp = polite_get("/draft/picks", {"year": year})
    out.write_text(resp.text, encoding="utf-8")
    print(f"{year}: HTTP {resp.status_code}, {len(resp.json())} picks saved")

2010: already saved, skipping
2011: already saved, skipping
2012: already saved, skipping
2013: already saved, skipping
2014: already saved, skipping
2015: already saved, skipping
2016: already saved, skipping
2017: already saved, skipping
2018: already saved, skipping
2019: already saved, skipping
2020: already saved, skipping
2021: already saved, skipping
2022: already saved, skipping
2023: already saved, skipping
2024: already saved, skipping
2025: already saved, skipping
2026: already saved, skipping


### 1.5 Transfer-portal entries

In [20]:
for year in PORTAL_YEARS:
    out = RAW_DIR / f"portal_{year}.json"
    if out.exists():
        print(f"{year}: already saved, skipping")
        continue
    resp = polite_get("/player/portal", {"year": year})
    out.write_text(resp.text, encoding="utf-8")
    print(f"{year}: HTTP {resp.status_code}, {len(resp.json())} entries saved")

2021: already saved, skipping
2022: already saved, skipping
2023: already saved, skipping
2024: already saved, skipping
2025: already saved, skipping
2026: already saved, skipping


### 1.6 Record what was collected
A small provenance file

In [21]:
manifest = {"source": BASE_URL, "user_agent": USER_AGENT, "pause_seconds": PAUSE_SECONDS, "files": []}
for path in sorted(RAW_DIR.glob("*.json")):
    manifest["files"].append({
        "file": path.name,
        "records": len(json.loads(path.read_text(encoding="utf-8"))),
        "bytes": path.stat().st_size,
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        "saved_utc": datetime.fromtimestamp(path.stat().st_mtime, timezone.utc).isoformat(timespec="seconds"),
    })
(DATA / "raw_manifest_cfbd.json").write_text(json.dumps(manifest, indent=2))
pd.DataFrame(manifest["files"])[["file", "records", "bytes", "saved_utc"]]

,file,records,bytes,saved_utc
0,draft_picks_2010.json,255,121217,2026-11-09T05:05:50+00:00
1,draft_picks_2011.json,254,120990,2026-11-09T05:05:51+00:00
2,draft_picks_2012.json,253,121440,2026-11-09T05:05:52+00:00
3,draft_picks_2013.json,254,121411,2026-11-09T05:05:54+00:00
4,draft_picks_2014.json,256,122756,2026-11-09T05:05:57+00:00
5,draft_picks_2015.json,256,122689,2026-11-09T05:05:57+00:00
6,draft_picks_2016.json,253,121370,2026-11-09T05:05:58+00:00
7,draft_picks_2017.json,253,121389,2026-11-09T05:06:00+00:00
8,draft_picks_2018.json,256,123486,2026-11-09T05:06:02+00:00
9,draft_picks_2019.json,254,122955,2026-11-09T05:06:03+00:00


## Section 2. Clean
the raw files saved above plus the nflverse CSVs already in `data/`.

### 2.1 Load the raw CFB draft picks

In [22]:
rows = []
for path in sorted(RAW_DIR.glob("draft_picks_*.json")):
    rows += json.loads(path.read_text(encoding="utf-8"))
cfbd_raw = pd.json_normalize(rows)     
print(cfbd_raw.shape)
print(list(cfbd_raw.columns))
cfbd_raw.head(3)

(4350, 24)
['collegeAthleteId', 'nflAthleteId', 'collegeId', 'collegeTeam', 'collegeConference', 'nflTeamId', 'nflTeam', 'year', 'overall', 'round', 'pick', 'name', 'position', 'height', 'weight', 'preDraftRanking', 'preDraftPositionRanking', 'preDraftGrade', 'hometownInfo.city', 'hometownInfo.state', 'hometownInfo.country', 'hometownInfo.latitude', 'hometownInfo.longitude', 'hometownInfo.countyFips']


,collegeAthleteId,nflAthleteId,collegeId,collegeTeam,collegeConference,nflTeamId,nflTeam,year,overall,round,...,weight,preDraftRanking,preDraftPositionRanking,preDraftGrade,hometownInfo.city,hometownInfo.state,hometownInfo.country,hometownInfo.latitude,hometownInfo.longitude,hometownInfo.countyFips
0,188934.0,25545,201,Oklahoma,Big 12,14,Los Angeles,2010,1,1,...,236.0,3.0,1.0,97.0,Oklahoma City,OK,USA,35.4729886,-97.5170536,40109
1,175300.0,25542,158,Nebraska,Big 12,8,Detroit,2010,2,1,...,307.0,2.0,2.0,97.0,Portland,OR,USA,45.5202471,-122.6741949,41051
2,2219195.0,25395,201,Oklahoma,Big 12,27,Tampa Bay,2010,3,1,...,295.0,1.0,1.0,98.0,None,None,None,None,None,None


### 2.2 Clean the CFB table

In [23]:
rename = {
    "year": "season",
    "overall": "pick",
    "name": "cfbd_name",                            
    "collegeConference": "college_conference",
    "preDraftGrade": "pre_draft_grade",
    "preDraftRanking": "pre_draft_rank",
    "preDraftPositionRanking": "pre_draft_pos_rank",
    "hometownInfo.state": "hometown_state",
    "hometownInfo.country": "hometown_country",
}
missing = [c for c in ("year", "overall") if c not in cfbd_raw.columns]
assert not missing, f"CFBD response is missing {missing}; columns found: {list(cfbd_raw.columns)}"


cfbd = cfbd_raw.drop(columns=["pick"], errors="ignore").rename(columns=rename)
cfbd = cfbd[[c for c in rename.values() if c in cfbd.columns]].copy()

for col in ("season", "pick", "pre_draft_grade", "pre_draft_rank", "pre_draft_pos_rank"):
    if col in cfbd.columns:
        cfbd[col] = pd.to_numeric(cfbd[col], errors="coerce")
for col in cfbd.select_dtypes(include=["object", "string"]).columns:
    cfbd[col] = cfbd[col].astype("string").str.strip().replace("", pd.NA)

n_loaded = len(cfbd)
cfbd = cfbd.dropna(subset=["season", "pick"])
n_missing = n_loaded - len(cfbd)
cfbd[["season", "pick"]] = cfbd[["season", "pick"]].astype(int)
n_dups = int(cfbd.duplicated(subset=["season", "pick"]).sum())
cfbd = cfbd.drop_duplicates(subset=["season", "pick"], keep="first")
print(f"{n_loaded} rows loaded; dropped {n_missing} with no season/pick and {n_dups} duplicate picks")
print(f"{len(cfbd)} CFBD picks kept")
cfbd.head(3)

4350 rows loaded; dropped 0 with no season/pick and 0 duplicate picks
4350 CFBD picks kept


,season,pick,cfbd_name,college_conference,pre_draft_grade,pre_draft_rank,pre_draft_pos_rank,hometown_state,hometown_country
0,2010,1,Sam Bradford,Big 12,97.0,3.0,1.0,OK,USA
1,2010,2,Ndamukong Suh,Big 12,97.0,2.0,2.0,OR,USA
2,2010,3,Gerald McCoy,Big 12,98.0,1.0,1.0,<NA>,<NA>


In [24]:
DRAFT_CSV = None    
PLAYERS_CSV = None 
def repo_root():
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / ".git").exists():
            return folder
    return here
def find_csv(keyword, override=None):
    if override:
        assert Path(override).exists(), f"{override} does not exist (current folder: {Path.cwd()})"
        return str(override)
    root = repo_root()
    hits, all_csv = [], []
    for folder, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if not d.startswith(".") and d not in ("node_modules", "__pycache__", "raw_cfbd")]
        for f in files:
            if f.lower().endswith(".csv"):
                all_csv.append(os.path.join(folder, f))
                if keyword in f.lower():
                    hits.append(os.path.join(folder, f))
    if not hits:
        print(f"Current folder: {Path.cwd()}")
        print(f"Searched under: {root}")
        print(f"CSV files found there ({len(all_csv)}):")
        for p in all_csv[:20]:
            print("   ", p)
        raise FileNotFoundError(f"No CSV with '{keyword}' in its name. Set DRAFT_CSV / PLAYERS_CSV above to the exact path.")
    hits.sort(key=lambda p: (len(p), p))   # shortest path first
    print(f"'{keyword}': using {hits[0]}" + (f"  (other matches ignored: {hits[1:]})" if len(hits) > 1 else ""))
    return hits[0]
draft_cols = {"season", "round", "pick", "team", "gsis_id", "pfr_player_id", "pfr_player_name",
              "position", "category", "side", "college", "age"}
draft = pd.read_csv(find_csv("draft_picks", DRAFT_CSV), usecols=lambda c: c in draft_cols, low_memory=False)
players = pd.read_csv(find_csv("players", PLAYERS_CSV), usecols=["gsis_id", "pfr_id", "birth_date"], low_memory=False)
print(len(draft), "draft picks;", len(players), "player rows")

'draft_picks': using /home/jovyan/NOTEBOOK/draft_picks.csv
'players': using /home/jovyan/NOTEBOOK/players (1).csv
12927 draft picks; 24867 player rows


### 2.4 get each rookie's age

In [25]:
NIL_START = pd.Timestamp(2021, 7, 1)
COVID_CLASSES = (2022, 2023)
MIN_AGE, MAX_AGE = 19, 30

log = {}   # counts shown at the end and used in the README text

# scope and duplicates
log["nflverse_picks_loaded"] = len(draft)
draft = draft[draft["season"].between(START_YEAR, END_YEAR)].copy()
draft = draft.drop_duplicates(subset=["season", "pick"], keep="first")
draft["season"] = draft["season"].astype(int)
log["picks_in_scope"] = len(draft)

players["birth_date"] = pd.to_datetime(players["birth_date"], errors="coerce")
by_gsis = players.dropna(subset=["gsis_id", "birth_date"]).drop_duplicates("gsis_id").set_index("gsis_id")["birth_date"]
by_pfr = players.dropna(subset=["pfr_id", "birth_date"]).drop_duplicates("pfr_id").set_index("pfr_id")["birth_date"]
draft["birth_date"] = draft["gsis_id"].map(by_gsis).fillna(draft["pfr_player_id"].map(by_pfr))

missing_by_class = draft["birth_date"].isna().groupby(draft["season"]).agg(["sum", "size"])
missing_by_class.columns = ["missing_birth_date", "picks"]
log["missing_birth_date"] = int(missing_by_class["missing_birth_date"].sum())
display(missing_by_class.T)

draft = draft.dropna(subset=["birth_date"]).copy()
anchor = pd.to_datetime({"year": draft["season"], "month": 4, "day": 30})
draft["age_apr30"] = ((anchor - draft["birth_date"]).dt.days / 365.25).round(2)
bad_age = ~draft["age_apr30"].between(MIN_AGE, MAX_AGE)
log["implausible_age_dropped"] = int(bad_age.sum())
draft = draft[~bad_age].copy()

pfr_age = pd.to_numeric(draft["age"], errors="coerce")
agree = ((draft["age_apr30"] - pfr_age).abs() <= 1)[pfr_age.notna()].mean()
print(f"Ages within 1 year of the source's own 'age' column: {agree:.1%}")
draft = draft.rename(columns={"age": "pfr_age_at_draft"})

anchor = pd.to_datetime({"year": draft["season"], "month": 4, "day": 30})
draft["era"] = (anchor >= NIL_START).map({False: "pre_nil", True: "post_nil"})
draft["covid_class"] = draft["season"].isin(COVID_CLASSES)
draft["position"] = draft["position"].astype("string").str.strip().str.upper()
draft["player_id"] = draft["gsis_id"].fillna(draft["pfr_player_id"])
log["rows_with_age"] = len(draft)
log

season,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
missing_birth_date,11,2,1,3,3,2,0,1,0,0,2,0,0,1,1,1,1
picks,255,254,253,254,256,256,253,253,256,254,255,259,262,259,257,257,257


Ages within 1 year of the source's own 'age' column: 99.3%


{'nflverse_picks_loaded': 12927,
 'picks_in_scope': 4350,
 'missing_birth_date': 29,
 'implausible_age_dropped': 1,
 'rows_with_age': 4320}

### 2.5 Join the CFB data and check the join

In [26]:
def squash(s):
    """lower-case letters only, so 'D.J. Moore Jr.' and 'DJ Moore' compare closer"""
    return s.astype("string").str.lower().str.replace(r"[^a-z]", "", regex=True)
merged = draft.merge(cfbd, on=["season", "pick"], how="left", validate="one_to_one", indicator=True)
merged["in_cfbd"] = merged["_merge"] == "both"
log["matched_in_cfbd"] = int(merged["in_cfbd"].sum())
print(f"{merged['in_cfbd'].mean():.1%} of rookies matched a CFBD pick")
print("Unmatched picks by class:")
print(merged.loc[~merged["in_cfbd"], "season"].value_counts().sort_index().to_dict())
both = merged[merged["in_cfbd"]]
a, b = squash(both["pfr_player_name"]), squash(both["cfbd_name"])
same = [bool(x) and bool(y) and (x == y or x in y or y in x) for x, y in zip(a.fillna(""), b.fillna(""))]
print(f"Names agree (loosely) for {sum(same) / max(len(same), 1):.1%} of matched rows")
merged = merged.drop(columns=["pfr_player_name", "cfbd_name", "birth_date", "_merge", "gsis_id", "pfr_player_id"])

100.0% of rookies matched a CFBD pick
Unmatched picks by class:
{}
Names agree (loosely) for 97.5% of matched rows


### 2.6 Transfer-portal counts
Only the number of entries per season is kept.

In [27]:
portal = pd.DataFrame(
    [{"season": int(p.stem.split("_")[1]),
      "portal_entries": len(json.loads(p.read_text(encoding="utf-8")))}
     for p in sorted(RAW_DIR.glob("portal_*.json"))]
)
portal

,season,portal_entries
0,2021,1770
1,2022,2273
2,2023,2502
3,2024,3378
4,2025,4499
5,2026,4484


### 2.7 Save the outputs

In [28]:
keep = ["player_id", "season", "round", "pick", "team", "position", "category", "side", "college",
        "college_conference", "pre_draft_grade", "pre_draft_rank", "pre_draft_pos_rank",
        "hometown_state", "hometown_country", "age_apr30", "pfr_age_at_draft", "era", "covid_class", "in_cfbd"]
tidy = (merged[[c for c in keep if c in merged.columns]]
        .rename(columns={"category": "position_category"})
        .sort_values(["season", "pick"]).reset_index(drop=True))

sample = (pd.concat([g.sample(min(15, len(g)), random_state=42) for _, g in tidy.groupby("era")])
          .sort_values(["season", "pick"]))

summary = (tidy.groupby("season")
           .agg(n_players=("age_apr30", "size"), median_age=("age_apr30", "median"),
                mean_age=("age_apr30", "mean"), era=("era", "first"), covid_class=("covid_class", "first"))
           .round(2).reset_index())

tidy.to_csv(CLEAN_DIR / "rookies_clean.csv", index=False)
sample.to_csv(SAMPLE_DIR / "rookies_sample.csv", index=False)
summary.to_csv(DATA / "summary_by_class.csv", index=False)
portal.to_csv(DATA / "portal_entries_by_season.csv", index=False)

log["tidy_rows"] = len(tidy)
print(f"{len(tidy)} tidy rows, {tidy['season'].nunique()} draft classes ({tidy['season'].min()}-{tidy['season'].max()})")
summary

4320 tidy rows, 17 draft classes (2010-2026)


,season,n_players,median_age,mean_age,era,covid_class
0,2010,244,22.70,22.61,pre_nil,False
1,2011,252,22.71,22.65,pre_nil,False
2,2012,252,22.67,22.77,pre_nil,False
3,2013,251,22.64,22.68,pre_nil,False
4,2014,253,22.58,22.60,pre_nil,False
5,2015,254,22.66,22.60,pre_nil,False
6,2016,253,22.54,22.52,pre_nil,False
7,2017,252,22.46,22.45,pre_nil,False
8,2018,256,22.40,22.40,pre_nil,False
9,2019,254,22.53,22.55,pre_nil,False


### 2.8 Look at the result
A quick look at the clean table

In [29]:
display(tidy.groupby("era")["age_apr30"].describe().round(2))
display(tidy.groupby("college_conference")["age_apr30"].agg(["size", "median"]).round(2)
        .sort_values("size", ascending=False).head(10))
sample.head(10)

,count,mean,std,min,25%,50%,75%,max
era,,,,,,,,
post_nil,1287.0,22.77,1.02,20.28,22.00,22.80,23.46,26.81
pre_nil,3033.0,22.55,0.89,19.99,21.93,22.55,23.17,28.54


,size,median
college_conference,,
SEC,990,22.42
Big Ten,717,22.55
ACC,606,22.58
Big 12,430,22.67
Pac-12,413,22.51
American Athletic,156,22.82
Mountain West,143,22.67
Conference USA,110,22.76
FBS Independents,100,22.47


,player_id,season,round,pick,team,position,position_category,side,college,college_conference,pre_draft_grade,pre_draft_rank,pre_draft_pos_rank,hometown_state,hometown_country,age_apr30,pfr_age_at_draft,era,covid_class,in_cfbd
44,00-0027658,2010,2,45,DEN,G,OL,O,Utah,Mountain West,74.0,81.0,5.0,UT,USA,23.44,23.0,pre_nil,False,True
52,00-0027661,2010,2,53,NWE,LB,LB,D,Florida,SEC,66.0,NaN,13.0,<NA>,<NA>,22.02,22.0,pre_nil,False,True
257,00-0027952,2011,1,14,STL,DE,DL,D,North Carolina,ACC,96.0,6.0,1.0,SC,USA,20.95,21.0,pre_nil,False,True
266,00-0027961,2011,1,23,PHI,G,OL,O,Baylor,Big 12,89.0,29.0,1.0,BC,Canada,26.48,26.0,pre_nil,False,True
789,00-0030579,2013,2,42,OAK,T,OL,O,Florida St.,ACC,89.0,30.0,6.0,<NA>,United Kingdom,24.35,24.0,pre_nil,False,True
807,00-0030528,2013,2,60,ATL,DB,DB,D,Southeastern Louisiana,Southland,78.0,67.0,10.0,<NA>,<NA>,24.49,24.0,pre_nil,False,True
1221,00-0031089,2014,7,224,PHI,DT,DL,D,Wisconsin,Big Ten,36.0,274.0,26.0,MN,USA,22.46,22.0,pre_nil,False,True
1518,00-0032380,2016,1,13,MIA,T,OL,O,Mississippi,SEC,94.0,2.0,1.0,FL,USA,21.74,22.0,pre_nil,False,True
1572,00-0033131,2016,3,67,DAL,DT,DL,D,Nebraska,Big Ten,76.0,78.0,14.0,MO,USA,21.06,21.0,pre_nil,False,True
1798,00-0033282,2017,2,40,CAR,WR,WR,O,Ohio St.,Big Ten,86.0,42.0,5.0,NY,USA,20.72,21.0,pre_nil,False,True
